# Chapter 3 EDA — Kepler Variable-Star Light Curves

This notebook reproduces the exploratory data analysis presented in **Chapter 3** of the thesis:
- Dataset overview and class distribution
- Light-curve length and completeness distributions
- Representative light curves per variability class
- Effect of gap injection on the flux distributions (Figure EDA)
- Lomb-Scargle period distributions by class


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from src.utils.period import lomb_scargle_peak_power
from src.data.gap_injection import inject_gaps
from src.features.extraction import extract_features, FEATURE_NAMES

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

DATA_DIR  = Path('../data/processed')
CLASS_NAMES = ['RRLYR', 'DSCT', 'EB', 'GDOR', 'SOL', 'ROT']
CLASS_COLORS = plt.cm.tab10(np.linspace(0, 0.6, 6))

## 1. Load Dataset

In [ ]:
# Load all preprocessed light curves
lcs = []
for fp in sorted(DATA_DIR.glob('*.parquet')):
    df = pd.read_parquet(fp)
    label = int(df['class_label'].iloc[0])
    lcs.append({
        'kic':   fp.stem,
        'time':  df['time'].values,
        'flux':  df['flux'].values,
        'label': label,
        'class': CLASS_NAMES[label],
        'n':     len(df),
    })

print(f'Total light curves: {len(lcs)}')

from collections import Counter
counts = Counter(lc['class'] for lc in lcs)
for cls in CLASS_NAMES:
    print(f'  {cls}: {counts.get(cls, 0)}')

## 2. Class Distribution (Table 3.1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cls_list  = [c for c in CLASS_NAMES if counts.get(c, 0) > 0]
cnt_list  = [counts[c] for c in cls_list]
bars = ax.bar(cls_list, cnt_list, color=CLASS_COLORS[:len(cls_list)])
for bar, cnt in zip(bars, cnt_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(cnt),
            ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Variability class')
ax.set_ylabel('Number of light curves')
ax.set_title('Dataset class distribution')
fig.tight_layout()
plt.show()

## 3. Representative Light Curves per Class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
axes = axes.flat

for i, cls in enumerate(CLASS_NAMES):
    examples = [lc for lc in lcs if lc['class'] == cls]
    if not examples:
        continue
    lc = examples[0]
    t, f = lc['time'], lc['flux']
    # Show first 500 cadences (~10 days)
    t_show = t[:500] - t[0]
    f_show = f[:500]
    axes[i].plot(t_show, f_show, 'k-', lw=0.6)
    axes[i].set_title(f'{cls} (KIC: {lc["kic"]})', fontsize=9)
    axes[i].set_xlabel('Time (days)', fontsize=8)
    axes[i].set_ylabel('Norm. flux', fontsize=8)
    axes[i].tick_params(labelsize=7)

fig.suptitle('Representative Kepler light curves by variability class', fontsize=11)
fig.tight_layout()
plt.show()

## 4. Effect of Gap Injection on Flux Distribution (Figure EDA)

In [ ]:
# Take one example from each class
examples = {cls: next((lc for lc in lcs if lc['class'] == cls), None) for cls in CLASS_NAMES}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flat

for i, cls in enumerate(CLASS_NAMES):
    lc = examples.get(cls)
    if lc is None:
        continue
    t, f = lc['time'], lc['flux']
    gapped, mask, _ = inject_gaps(f, p=0.30, seed=42)

    t_obs = t[mask]
    f_obs = f[mask]
    t_gap = t[~mask]

    t_show = t[:500] - t[0]
    g_show = gapped[:500]
    f_show = f[:500]

    axes[i].plot(t_show, f_show, 'k-', lw=0.4, alpha=0.4, label='Complete')
    axes[i].plot(t_show, g_show, 'b.', ms=1.5, label='Observed (30% gap)')

    # Shade gaps
    in_gap = False
    for idx in range(500):
        if not mask[idx] and not in_gap:
            gs = t_show[idx]; in_gap = True
        elif mask[idx] and in_gap:
            axes[i].axvspan(gs, t_show[idx], color='orange', alpha=0.2)
            in_gap = False

    axes[i].set_title(f'{cls}', fontsize=9)
    axes[i].set_xlabel('Time (days)', fontsize=8)
    axes[i].set_ylabel('Norm. flux', fontsize=8)
    if i == 0:
        axes[i].legend(fontsize=7)

fig.suptitle('Effect of 30% gap injection on Kepler light curves', fontsize=11)
fig.tight_layout()
plt.show()

## 5. Period Distributions by Class

In [ ]:
print('Estimating periods for all light curves (may take a few minutes)...')
periods = []
for lc in lcs:
    p, power = lomb_scargle_peak_power(lc['time'], lc['flux'])
    periods.append({'class': lc['class'], 'period': p, 'ls_power': power})

period_df = pd.DataFrame(periods)

fig, ax = plt.subplots(figsize=(9, 5))
for i, cls in enumerate(CLASS_NAMES):
    sub = period_df[period_df['class'] == cls]['period'].values
    ax.hist(np.log10(sub[sub > 0]), bins=30, density=True, alpha=0.5,
            color=CLASS_COLORS[i], label=cls)
ax.set_xlabel('log₁₀ Period (days)')
ax.set_ylabel('Density')
ax.set_title('Lomb-Scargle period distributions by variability class')
ax.legend()
fig.tight_layout()
plt.show()

## 6. Feature Correlation Matrix

Understanding which features are correlated helps explain why certain imputation artefacts propagate into misclassification.

In [ ]:
import seaborn as sns

print('Extracting features for all light curves...')
feats = np.stack([extract_features(lc['time'], lc['flux']) for lc in lcs])
labels_arr = np.array([lc['label'] for lc in lcs])

feat_df = pd.DataFrame(feats, columns=FEATURE_NAMES)
corr = feat_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, ax=ax,
            xticklabels=FEATURE_NAMES, yticklabels=FEATURE_NAMES)
ax.set_title('Feature correlation matrix (35 features)', fontsize=11)
fig.tight_layout()
plt.show()